# Machine Learning - Assignment 2: Classification Models Comparison
**M.Tech (AIML) - BITS Pilani WILP**

Student: NAITIK CHATTERJEE | BITS ID: 2026CT05027

This notebook runs the full assignment: dataset exploration, preprocessing, five classifiers (Logistic Regression, Decision Tree, kNN, Naive Bayes, Random Forest), six evaluation metrics, graphs, and it **generates the repository files** (`README.md`, `requirements.txt`, `test_data.csv`, saved models) and a PDF-ready report from the results of the run.

**Configuration.** After creating your GitHub repository, paste its link into `GITHUB_REPO_URL` below and re-run the README and report cells. The other settings can stay as they are.

In [ ]:
GITHUB_REPO_URL = ""
RANDOM_STATE = 42
TEST_SIZE = 0.20
OPENML_DATA_ID = 1590
TARGET = "class"
POSITIVE_CLASS = ">50K"
OUT_DIR = "project-folder"

import os
for sub in ["", "model", "images"]:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
print("Output folder:", OUT_DIR)

In [ ]:
import warnings, time, json, base64
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, confusion_matrix, roc_curve)

try:
    from IPython.display import display
except Exception:
    display = print

sns.set_theme(style="whitegrid")
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

## Step 1: Dataset choice

**Dataset:** Adult Census Income (UCI Machine Learning Repository, also hosted on OpenML).
**Task:** binary classification - predict whether a person earns more than 50K USD per year from census attributes.

**Why this dataset:** it has 14 mixed numeric and categorical features (above the 12 required), roughly 48,000 rows (above the 1,000 required), real missing values that must be handled, and a clear class imbalance (about 24% positive), which makes accuracy alone misleading and makes metrics like AUC and MCC meaningful.

In [ ]:
from sklearn.datasets import fetch_openml

UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
UCI_COLUMNS = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status", "occupation",
               "relationship", "race", "sex", "capital-gain", "capital-loss", "hours-per-week",
               "native-country", "class"]
try:
    bunch = fetch_openml(data_id=OPENML_DATA_ID, as_frame=True)
    df = bunch.frame.copy()
    DATASET_SOURCE = f"UCI Adult Census Income, loaded via OpenML (data id {OPENML_DATA_ID})"
except Exception as err:
    print("OpenML download failed (" + type(err).__name__ + "), reading directly from UCI instead.")
    df = pd.read_csv(UCI_URL, header=None, names=UCI_COLUMNS, na_values=" ?", skipinitialspace=True)
    DATASET_SOURCE = "UCI Machine Learning Repository - Adult (" + UCI_URL + ")"

for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object).where(df[col].notna(), np.nan)
        df[col] = df[col].map(lambda v: v.strip() if isinstance(v, str) else v).replace({"?": np.nan})

print("Source:", DATASET_SOURCE)
print("Shape:", df.shape)
display(df.head())

In [ ]:
feature_cols = [c for c in df.columns if c != TARGET]
numeric_features = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
categorical_features = [c for c in feature_cols if c not in numeric_features]

print(f"Rows: {len(df):,} | Features: {len(feature_cols)} | Target: '{TARGET}'")
print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
assert len(feature_cols) >= 12, "assignment needs at least 12 features"
assert len(df) >= 1000, "assignment needs at least 1000 instances"

print("\nNumeric feature summary:")
display(df[numeric_features].describe().round(2))
print("\nCategorical feature cardinality:")
display(df[categorical_features].nunique().to_frame("unique values").T)

In [ ]:
target_counts = df[TARGET].astype(str).str.strip().value_counts()
target_pct = (target_counts / target_counts.sum() * 100).round(2)
print(pd.DataFrame({"count": target_counts, "percent": target_pct}).to_string())
imbalance_ratio = target_counts.max() / target_counts.min()
print(f"\nImbalance ratio (majority : minority) = {imbalance_ratio:.2f} : 1")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
sns.barplot(x=target_counts.index, y=target_counts.values, hue=target_counts.index, legend=False, ax=ax[0])
ax[0].set_title("Class distribution (count)"); ax[0].set_xlabel(TARGET); ax[0].set_ylabel("rows")
ax[1].pie(target_counts.values, labels=target_counts.index, autopct="%1.1f%%", startangle=90)
ax[1].set_title("Class distribution (share)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "images", "class_distribution.png"), dpi=130)
plt.show()

## Step 2: Data preprocessing
- Missing values: counted here, then imputed **inside the pipeline** (median for numeric, most frequent for categorical) so the imputer is fitted on training data only.
- Duplicates: counted and removed.
- Categorical features: one-hot encoding. Numeric features: standard scaling.
- Split: 80:20, stratified on the target, fixed random seed.
- All preprocessing steps are part of a scikit-learn `Pipeline`, so they are **fitted only on the training data** and then applied to the test data.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0]
print("Missing values per column (before handling):")
print(missing.to_string() if len(missing) else "none")
print("Total missing cells:", int(df.isna().sum().sum()))

n_dup = int(df.duplicated().sum())
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"\nDuplicate rows found and removed: {n_dup} ({n_before:,} -> {len(df):,} rows)")

le = LabelEncoder()
y = le.fit_transform(df[TARGET].astype(str).str.strip())
class_names = [str(c) for c in le.classes_]
n_classes = len(class_names)
pos_label = class_names.index(POSITIVE_CLASS) if POSITIVE_CLASS in class_names else (1 if n_classes == 2 else None)
X = df[feature_cols].copy()
print("Target encoding:", {c: int(i) for c, i in zip(class_names, le.transform(class_names))}, "| positive class index:", pos_label)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

print(f"Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")
dist = pd.DataFrame({
    "full data %": pd.Series(y).value_counts(normalize=True).sort_index() * 100,
    "train %": pd.Series(y_train).value_counts(normalize=True).sort_index() * 100,
    "test %": pd.Series(y_test).value_counts(normalize=True).sort_index() * 100,
}).round(2)
dist.index = class_names
print("\nClass distribution is maintained across the split:")
print(dist.to_string())


def make_preprocessor():
    numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    categorical_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                                 ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", numeric_pipe, numeric_features),
                              ("cat", categorical_pipe, categorical_features)])

print("\nPreprocessing pipeline defined (fitted on the training data only, inside each model pipeline).")

## Step 3: Classification models and evaluation metrics
Five models on the same dataset and the same split. Metrics for each: Accuracy, AUC, Precision, Recall, F1 and MCC. For a binary target, precision, recall and F1 are for the positive class; for a multi-class target they would be macro-averaged.

In [ ]:
model_defs = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE),
    "kNN": KNeighborsClassifier(n_neighbors=15, weights="distance", n_jobs=-1),
    "Naive Bayes": GaussianNB(),
    "Random Forest (Ensemble)": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE),
}
model_configs = {name: ", ".join(f"{k}={v}" for k, v in m.get_params().items()
                                 if k in ("C", "max_iter", "max_depth", "min_samples_leaf", "n_neighbors", "weights",
                                          "n_estimators", "var_smoothing", "random_state"))
                 for name, m in model_defs.items()}
for name, cfg in model_configs.items():
    print(f"{name}: {cfg}")

In [ ]:
avg = "binary" if n_classes == 2 else "macro"
kw = {"pos_label": pos_label} if n_classes == 2 else {}
rng = np.random.RandomState(RANDOM_STATE)
train_sample_idx = rng.choice(len(X_train), size=min(5000, len(X_train)), replace=False)

fitted, results, preds, probas, extra = {}, [], {}, {}, {}
for name, clf in model_defs.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("model", clf)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    fit_time = time.time() - t0

    t0 = time.time()
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)
    pred_time = time.time() - t0

    if n_classes == 2:
        auc = roc_auc_score(y_test, y_proba[:, pos_label])
    else:
        auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")

    train_acc = accuracy_score(y_train[train_sample_idx], pipe.predict(X_train.iloc[train_sample_idx]))
    results.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": auc,
        "Precision": precision_score(y_test, y_pred, average=avg, zero_division=0, **kw),
        "Recall": recall_score(y_test, y_pred, average=avg, zero_division=0, **kw),
        "F1": f1_score(y_test, y_pred, average=avg, zero_division=0, **kw),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })
    fitted[name], preds[name], probas[name] = pipe, y_pred, y_proba
    extra[name] = {"train_accuracy": float(train_acc), "fit_time": fit_time, "predict_time": pred_time}
    joblib.dump(pipe, os.path.join(OUT_DIR, "model", name.split(" (")[0].lower().replace(" ", "_") + ".joblib"))
    print(f"{name:26s} done | fit {fit_time:6.1f}s | predict {pred_time:5.1f}s | test accuracy {results[-1]['Accuracy']:.4f}")

In [ ]:
results_df = pd.DataFrame(results)
metric_cols = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
print("Comparison table (test set):")
display(results_df.round(4))
results_df.round(6).to_csv(os.path.join(OUT_DIR, "results.csv"), index=False)

print("\nBest model per metric:")
for m in metric_cols:
    best = results_df.loc[results_df[m].idxmax()]
    print(f"  {m:10s}: {best['ML Model Name']} ({best[m]:.4f})")

### Visualisation of all metrics

In [ ]:
long_df = results_df.melt(id_vars="ML Model Name", value_vars=metric_cols, var_name="Metric", value_name="Score")
plt.figure(figsize=(12, 5))
sns.barplot(data=long_df, x="Metric", y="Score", hue="ML Model Name")
plt.ylim(0, 1.05); plt.title("All evaluation metrics by model (test set)")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8, ncol=5)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "images", "metrics_bar.png"), dpi=130)
plt.show()

plt.figure(figsize=(8, 3.6))
sns.heatmap(results_df.set_index("ML Model Name")[metric_cols], annot=True, fmt=".3f", cmap="YlGnBu", vmin=0, vmax=1)
plt.title("Metric heatmap"); plt.ylabel("")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "images", "metrics_heatmap.png"), dpi=130)
plt.show()

In [ ]:
if n_classes == 2:
    plt.figure(figsize=(6.5, 5.5))
    for name in model_defs:
        fpr, tpr, _ = roc_curve(y_test, probas[name][:, pos_label])
        plt.plot(fpr, tpr, label=f"{name} (AUC={results_df.set_index('ML Model Name').loc[name, 'AUC']:.3f})")
    plt.plot([0, 1], [0, 1], "k--", lw=1)
    plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("ROC curves")
    plt.legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "images", "roc_curves.png"), dpi=130)
    plt.show()

fig, axes = plt.subplots(2, 3, figsize=(13, 7.5))
for ax, name in zip(axes.ravel(), model_defs):
    sns.heatmap(confusion_matrix(y_test, preds[name]), annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for ax in axes.ravel()[len(model_defs):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "images", "confusion_matrices.png"), dpi=130)
plt.show()

## Observations on model performance
The observation text below is generated from the numbers of this run. **Read it and rewrite it in your own words** (cell below) before you submit.

In [ ]:
static_notes = {
    "Logistic Regression": "It learns one linear decision boundary on the scaled and one-hot encoded features, so it is fast and stable but cannot capture feature interactions.",
    "Decision Tree": "A single tree (depth limited to 8) captures non-linear rules and interactions and is easy to interpret, but one tree is high-variance and sensitive to the training sample.",
    "kNN": "It classifies by the 15 nearest training points, so it depends on feature scaling and loses discriminative power in a high-dimensional one-hot space; it also has to compare every test row with all training rows at prediction time.",
    "Naive Bayes": "Gaussian Naive Bayes assumes independent, normally distributed features; one-hot columns are neither, so its probability estimates are often less reliable than those of the other models.",
    "Random Forest (Ensemble)": "Averaging 300 decorrelated trees reduces the variance of a single tree and captures non-linear interactions, which typically gives the most robust results on tabular data.",
}
res = results_df.set_index("ML Model Name")
rank_mcc = res["MCC"].rank(ascending=False, method="min").astype(int)
pos_name = class_names[pos_label] if pos_label is not None else "each class"

observations = {}
for name in model_defs:
    r = res.loc[name]
    gap = extra[name]["train_accuracy"] - r["Accuracy"]
    if r["Precision"] > r["Recall"] + 0.03:
        pr = f"Precision ({r['Precision']:.3f}) is higher than recall ({r['Recall']:.3f}), so it is cautious about predicting '{pos_name}' and misses some true cases."
    elif r["Recall"] > r["Precision"] + 0.03:
        pr = f"Recall ({r['Recall']:.3f}) is higher than precision ({r['Precision']:.3f}), so it finds most '{pos_name}' cases but with more false alarms."
    else:
        pr = f"Precision ({r['Precision']:.3f}) and recall ({r['Recall']:.3f}) are well balanced."
    if gap > 0.05:
        fit = f"Training accuracy ({extra[name]['train_accuracy']:.3f}) is clearly above test accuracy, which indicates overfitting."
    else:
        fit = f"Training accuracy ({extra[name]['train_accuracy']:.3f}) is close to test accuracy, so it generalises well."
    observations[name] = (f"Accuracy {r['Accuracy']:.3f}, AUC {r['AUC']:.3f}, F1 {r['F1']:.3f}, MCC {r['MCC']:.3f} "
                          f"(rank {rank_mcc[name]} of {len(res)} by MCC). {pr} {fit} {static_notes[name]}")

ranked = res.sort_values(["MCC", "AUC"], ascending=False)
winner, runner_up = ranked.index[0], ranked.index[1]
wins = int(sum(res[m].idxmax() == winner for m in metric_cols))
winner_text = (f"{winner} is the overall winner on this dataset: it ranks first by MCC ({res.loc[winner, 'MCC']:.3f}, AUC {res.loc[winner, 'AUC']:.3f}) "
               f"and is the best model on {wins} of {len(metric_cols)} metrics; the runner-up is {runner_up} "
               f"(MCC {res.loc[runner_up, 'MCC']:.3f}). Because the classes are imbalanced "
               f"({target_pct.min():.1f}% minority class), accuracy alone is optimistic, so MCC and AUC were used as the main criteria.")

for name, text in observations.items():
    print(f"- {name}: {text}\n")
print("Overall winner:", winner_text)

## Repository files: test data, README, requirements, report

In [ ]:
test_out = X_test.copy()
test_out[TARGET] = [class_names[i] for i in y_test]
test_out.to_csv(os.path.join(OUT_DIR, "test_data.csv"), index=False)

with open(os.path.join(OUT_DIR, "requirements.txt"), "w") as f:
    f.write("scikit-learn\nnumpy\npandas\nmatplotlib\nseaborn\njoblib\n")

print("test_data.csv:", test_out.shape, "| requirements.txt written")

In [ ]:
def md_table(df, digits=4):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(f"{v:.{digits}f}" if isinstance(v, (float, np.floating)) else str(v) for v in row) + " |")
    return "\n".join(lines)

obs_df = pd.DataFrame({"ML Model Name": list(observations), "Observation about model performance": list(observations.values())})
repo_line = GITHUB_REPO_URL if GITHUB_REPO_URL else "(add your GitHub repository link here)"
type_line = "binary classification" if n_classes == 2 else f"{n_classes}-class classification"
feat_tbl = pd.DataFrame({"Feature type": ["Numeric", "Categorical"],
                         "Count": [len(numeric_features), len(categorical_features)],
                         "Features": [", ".join(numeric_features), ", ".join(categorical_features)]})

readme = f"""# Classification Models Comparison - ML Assignment 2

**Student:** NAITIK CHATTERJEE | **BITS ID:** 2026CT05027

## 1. Problem statement
Build and compare five machine-learning classifiers (Logistic Regression, Decision Tree, kNN, Naive Bayes and Random Forest) on one public dataset, and evaluate them with Accuracy, AUC, Precision, Recall, F1 and MCC. The task is {type_line}: predict whether a person's yearly income is above 50K USD ('{POSITIVE_CLASS}') from census attributes.

## 2. Dataset description
- **Source:** {DATASET_SOURCE}
- **Why chosen:** mixed numeric and categorical features, real missing values, and a clear class imbalance make the comparison of metrics meaningful.
- **Instances:** {n_before:,} rows ({len(df):,} after removing {n_dup} duplicates) | **Features:** {len(feature_cols)} | **Target:** `{TARGET}` ({', '.join(class_names)})
- **Class distribution:** {', '.join(f'{k}: {v:,} ({target_pct[k]}%)' for k, v in target_counts.items())} (imbalance ratio {imbalance_ratio:.2f}:1)
- **Missing values:** {int(missing.sum())} cells in {len(missing)} columns ({', '.join(missing.index) if len(missing) else 'none'}), imputed inside the pipeline.

{md_table(feat_tbl)}

![Class distribution](images/class_distribution.png)

### Preprocessing
1. Duplicate rows removed ({n_dup}).
2. Missing values imputed: median for numeric, most frequent for categorical.
3. Categorical features one-hot encoded; numeric features standardised.
4. Stratified {int((1-TEST_SIZE)*100)}:{int(TEST_SIZE*100)} train-test split ({len(X_train):,} / {len(X_test):,}) with random seed {RANDOM_STATE}.
5. Every preprocessing step lives inside a scikit-learn `Pipeline`, so it is fitted on the training data only.

## 3. GitHub repository link
{repo_line}

```
project-folder/
|-- requirements.txt
|-- README.md
|-- test_data.csv
|-- model/   (notebook + saved models: *.joblib)
```

## 4. Models used
Model configurations:

{chr(10).join(f'- **{n}:** {c}' for n, c in model_configs.items())}

### Comparison table (test set)
{md_table(results_df)}

![All metrics](images/metrics_bar.png)
![Metric heatmap](images/metrics_heatmap.png)
{"![ROC curves](images/roc_curves.png)" if n_classes == 2 else ""}
![Confusion matrices](images/confusion_matrices.png)

### Observations
{md_table(obs_df)}

**Overall winner for this dataset:** {winner_text}

## 5. How to run
```
pip install -r requirements.txt
jupyter notebook model/ML_Assignment_2.ipynb     # Run All
```
The notebook downloads the dataset, trains the five models, prints the metrics, draws the graphs and rewrites this README, `test_data.csv` and the saved models.
"""
with open(os.path.join(OUT_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme)
print("README.md written (", len(readme.split()), "words )")

In [ ]:
import html as _html


def b64_img(path):
    with open(path, "rb") as fh:
        return "data:image/png;base64," + base64.b64encode(fh.read()).decode()


def img_tag(fname, caption):
    p = os.path.join(OUT_DIR, "images", fname)
    return f'<figure><img src="{b64_img(p)}" style="max-width:100%"><figcaption>{caption}</figcaption></figure>' if os.path.exists(p) else ""


shot_path = os.path.join(OUT_DIR, "images", "bits_lab_screenshot.png")
if os.path.exists(shot_path):
    shot_html = img_tag("bits_lab_screenshot.png", "Execution on BITS Virtual Lab")
else:
    shot_html = ('<p style="color:#b00"><b>Screenshot missing:</b> save your BITS Virtual Lab screenshot as '
                 f'<code>{OUT_DIR}/images/bits_lab_screenshot.png</code> and re-run this cell.</p>')

link_html = (f'<a href="{GITHUB_REPO_URL}">{GITHUB_REPO_URL}</a>' if GITHUB_REPO_URL
             else '<span style="color:#b00">(set GITHUB_REPO_URL in the configuration cell and re-run)</span>')
obs_rows = "".join(f"<tr><td><b>{_html.escape(n)}</b></td><td>{_html.escape(t)}</td></tr>" for n, t in observations.items())

report = f"""<!doctype html><html><head><meta charset="utf-8"><title>ML Assignment 2 - 2026CT05027</title>
<style>body{{font-family:Arial,sans-serif;max-width:900px;margin:24px auto;padding:0 16px;color:#222;line-height:1.45}}
table{{border-collapse:collapse;width:100%;margin:8px 0}}td,th{{border:1px solid #bbb;padding:5px 8px;font-size:13px;vertical-align:top}}
th{{background:#eee}}h2{{border-bottom:1px solid #ccc;padding-bottom:3px;margin-top:28px}}figure{{margin:12px 0}}figcaption{{font-size:12px;color:#555}}
code{{background:#f2f2f2;padding:1px 4px}}</style></head><body>
<h1>Classification Models Comparison - ML Assignment 2</h1>
<p><b>Student:</b> NAITIK CHATTERJEE &nbsp; | &nbsp; <b>BITS ID:</b> 2026CT05027</p>
<h2>GitHub Repository Link</h2><p>{link_html}</p>
<h2>1. Problem statement</h2>
<p>Build and compare five classifiers (Logistic Regression, Decision Tree, kNN, Naive Bayes, Random Forest) on one public dataset and evaluate them with Accuracy, AUC, Precision, Recall, F1 and MCC. The task is {type_line}: predict whether yearly income is above 50K USD ('{_html.escape(POSITIVE_CLASS)}') from census attributes.</p>
<h2>2. Dataset description</h2>
<ul><li><b>Source:</b> {_html.escape(DATASET_SOURCE)}</li>
<li><b>Instances:</b> {n_before:,} rows ({len(df):,} after removing {n_dup} duplicates) | <b>Features:</b> {len(feature_cols)} | <b>Target:</b> {TARGET} ({_html.escape(', '.join(class_names))})</li>
<li><b>Class distribution:</b> {_html.escape(', '.join(f'{k}: {v:,} ({target_pct[k]}%)' for k, v in target_counts.items()))} (imbalance {imbalance_ratio:.2f}:1)</li>
<li><b>Missing values:</b> {int(missing.sum())} cells in {len(missing)} columns, imputed inside the pipeline.</li></ul>
{feat_tbl.to_html(index=False, border=0)}
{img_tag("class_distribution.png", "Class distribution")}
<p><b>Preprocessing:</b> duplicates removed; median / most-frequent imputation; one-hot encoding; standard scaling; stratified {int((1-TEST_SIZE)*100)}:{int(TEST_SIZE*100)} split with seed {RANDOM_STATE}; all steps fitted on the training data only (scikit-learn Pipeline).</p>
<h2>3. Models used - comparison table (test set)</h2>
{results_df.round(4).to_html(index=False, border=0)}
{img_tag("metrics_bar.png", "All metrics by model")}
{img_tag("metrics_heatmap.png", "Metric heatmap")}
{img_tag("roc_curves.png", "ROC curves")}
{img_tag("confusion_matrices.png", "Confusion matrices")}
<h2>4. Observations</h2>
<table><tr><th>ML Model Name</th><th>Observation about model performance</th></tr>{obs_rows}</table>
<p><b>Overall winner for this dataset:</b> {_html.escape(winner_text)}</p>
<h2>5. Screenshot - assignment execution on BITS Virtual Lab</h2>{shot_html}
</body></html>"""
with open(os.path.join(OUT_DIR, "report.html"), "w", encoding="utf-8") as f:
    f.write(report)
print("report.html written. Open it in a browser and use Print -> Save as PDF for the submission PDF.")

In [ ]:
import shutil
print("Repository contents:")
for root, _, files in os.walk(OUT_DIR):
    for fn in sorted(files):
        print("  ", os.path.join(root, fn))

todo = []
if not GITHUB_REPO_URL:
    todo.append("Set GITHUB_REPO_URL in the configuration cell, then re-run the README and report cells.")
if not os.path.exists(os.path.join(OUT_DIR, "images", "bits_lab_screenshot.png")):
    todo.append(f"Save the BITS Virtual Lab screenshot as {OUT_DIR}/images/bits_lab_screenshot.png, then re-run the report cell.")
todo.append("Save a copy of this notebook into " + OUT_DIR + "/model/ and rewrite the observations in your own words.")
print("\nStill to do:")
for t in todo:
    print(" -", t)

shutil.make_archive(OUT_DIR, "zip", ".", OUT_DIR)
print("\nZipped:", OUT_DIR + ".zip")
try:
    from google.colab import files as colab_files
    colab_files.download(OUT_DIR + ".zip")
except Exception:
    print("(not running on Colab - download the zip from the file browser)")